# Federated Learning: FedAvg, FedProx, SCAFFOLD, and Secure Aggregation

This notebook explores foundational decentralized and privacy-preserving optimization algorithms:
1. **FedAvg**: Weighted model parameter averaging across edge client devices.
2. **FedProx**: Proximal regularization damping local divergence on heterogeneous non-IID partitions.
3. **SCAFFOLD**: Control variates estimating and correcting client drift.
4. **SecAgg**: Cryptographic pairwise additive mask cancellation enabling private aggregation.

**Decision Boundary (When NOT to Use)**:
- Do not use FedAvg with standard Batch Normalization across heterogeneous non-IID edge devices; use GroupNorm or LayerNorm to prevent running statistic divergence.
- Do not use simple pairwise SecAgg without secret-sharing threshold drop-out recovery in unreliable mobile networks where clients drop out mid-round.

**Topic Bridges & Connects To**:
- `05_continual_learning_ewc_synaptic_intelligence_lwf_and_replay` (Sequential domain shifts)
- `07_active_learning_bald_coreset_and_badge` (Selective data acquisition on clients)

## 2. Mathematical Formulations & Theoretical Foundations

- **FedAvg Parameter Aggregation**:
  $$w_{\text{global}} = \sum_{k=1}^K \frac{n_k}{N} w_k, \quad \mathcal{O}(K \cdot D)$$
- **FedProx Proximal Gradient Step**:
  $$w \leftarrow w - \eta \left( \nabla F_k(w) + \mu (w - w_{\text{global}}) \right), \quad \mathcal{O}(D)$$
- **SCAFFOLD Drift-Corrected Gradient**:
  $$g_{\text{scaffold}} = g_i - c_i + c_{\text{global}}, \quad \mathcal{O}(D)$$
- **SecAgg Pairwise Additive Mask Cancellation**:
  $$\sum_{i=1}^K y_i = \sum_{i=1}^K w_i + \sum_{i=1}^K \sum_{j \neq i} s_{i,j} = \sum_{i=1}^K w_i, \quad \mathcal{O}(K \cdot D)$$

In [ ]:
import math
from pathlib import Path
import torch
import torch.nn.functional as F
from safetensors.torch import load_file

torch.manual_seed(42)
REPO_ROOT = next(p for p in Path.cwd().parents if (p / "Cargo.toml").exists()) if not (Path.cwd() / "Cargo.toml").exists() else Path.cwd()
GOLDEN_PATH = REPO_ROOT / "weights" / "08_learning_paradigms_rl_and_alignment" / "01_self_semi_meta_continual_federated" / "06_federated_learning_fedavg_fedprox_scaffold_and_secagg.safetensors"
golden = load_file(str(GOLDEN_PATH))
print(f"Loaded {len(golden)} tensors from golden checkpoint.")

In [ ]:
# 1. FedAvg Weighted Parameter Aggregation
def fedavg_aggregate(client_weights: torch.Tensor, client_counts: torch.Tensor) -> torch.Tensor:
    weights_ratio = client_counts / client_counts.sum()
    return (weights_ratio.unsqueeze(1) * client_weights).sum(dim=0)

fedavg_calc = fedavg_aggregate(golden["client_weights"], golden["client_counts"])
diff_fedavg = torch.max(torch.abs(fedavg_calc - golden["fedavg_aggregated_ref"])).item()
print(f"FedAvg Aggregated max abs error: {diff_fedavg:.2e}")
assert diff_fedavg < 1e-5

In [ ]:
# 2. FedProx Proximal Gradient Step
def fedprox_gradient_step(
    w_local: torch.Tensor,
    w_global: torch.Tensor,
    grad_local: torch.Tensor,
    eta: float = 0.05,
    mu: float = 0.1,
) -> torch.Tensor:
    prox_grad = grad_local + mu * (w_local - w_global)
    return w_local - eta * prox_grad

w_prox_calc = fedprox_gradient_step(golden["w_local"], golden["w_global"], golden["grad_local"], eta=0.05, mu=0.1)
diff_prox = torch.max(torch.abs(w_prox_calc - golden["w_prox_ref"])).item()
print(f"FedProx Step max abs error: {diff_prox:.2e}")
assert diff_prox < 1e-5

In [ ]:
# 3. SCAFFOLD Client Drift Correction
def scaffold_correct_gradient(
    grad_i: torch.Tensor,
    c_i: torch.Tensor,
    c_global: torch.Tensor,
) -> torch.Tensor:
    return grad_i - c_i + c_global

scaffold_calc = scaffold_correct_gradient(golden["grad_i"], golden["c_i"], golden["c_global"])
diff_scaf = torch.max(torch.abs(scaffold_calc - golden["scaffold_corrected_ref"])).item()
print(f"SCAFFOLD Corrected max abs error: {diff_scaf:.2e}")
assert diff_scaf < 1e-5

In [ ]:
# 4. SecAgg Pairwise Mask Cancellation & Anti-Pattern Contrast
def secagg_aggregate_sum(masked_models: torch.Tensor) -> torch.Tensor:
    return masked_models.sum(dim=0)

secagg_calc = secagg_aggregate_sum(golden["masked_models"])
diff_sec = torch.max(torch.abs(secagg_calc - golden["secagg_unmasked_sum_ref"])).item()
assert diff_sec < 1e-5

# Misuse Anti-Pattern: Unweighted parameter aggregation introducing size-bias
client_w = torch.tensor([[10.0, 10.0], [0.0, 0.0]])
counts = torch.tensor([1000.0, 1.0])
# Correct sample-weighted: heavily weights client 0 (~9.99)
correct_agg = fedavg_aggregate(client_w, counts)
# Incorrect unweighted naive mean: equal weights (5.0), drowning out 99.9% of user data
naive_mean = client_w.mean(dim=0)
bias_error = torch.abs(correct_agg - naive_mean).max().item()
assert bias_error > 4.5, "Anti-Pattern: Unweighted aggregation must produce extreme sampling bias"

print("=" * 65)
print(f"{'Operator':<32} | {'Max Abs Error':<15} | Status")
print("-" * 65)
print(f"{'FedAvg Aggregation':<32} | {diff_fedavg:<15.2e} | PASSED")
print(f"{'FedProx Proximal Step':<32} | {diff_prox:<15.2e} | PASSED")
print(f"{'SCAFFOLD Drift Correction':<32} | {diff_scaf:<15.2e} | PASSED")
print(f"{'SecAgg Unmasked Sum':<32} | {diff_sec:<15.2e} | PASSED")
print("=" * 65)

## 8. Summary, Key Takeaways & Cross-Domain Generalisation

- [x] **FedAvg**: Distributed coordinate descent trading local edge computation for global communication rounds.
- [x] **FedProx**: Proximal point regularization bounds client divergence on statistical non-IID and systems stragglers.
- [x] **SCAFFOLD**: Control variates maintain global drift estimates, accelerating non-IID convergence.
- [x] **SecAgg**: Zero-sum pairwise random masks provide cryptographic privacy without exposing individual model updates to the aggregation server.

**Cross-Domain Generalisation**:
Federated paradigms enable cross-silo training in healthcare (training medical diagnostic models across hospitals without sharing patient MRIs), mobile keyboard next-word prediction (learning user typing habits locally on mobile devices), and predictive fleet maintenance (training failure prediction models across commercial aircraft engines).

See curriculum index: [INDEX_BY_GOAL](../../../INDEX_BY_GOAL.md).